# Multi-Agent Evaluation

Compare and evaluate different agent frameworks and configurations using MLFlow.

## Contents
1. **Setup** - Initialize multiple adapters
2. **Define Tasks** - Create evaluation tasks
3. **Run Experiments** - Execute agents with tracking
4. **Compare Results** - Analyze performance in MLFlow


In [ ]:
# Setup - Initialize adapters
from agentic_assistants import AgenticConfig
from agentic_assistants.adapters import CrewAIAdapter, LangGraphAdapter
from agentic_assistants.core import MLFlowTracker
import mlflow

config = AgenticConfig()
tracker = MLFlowTracker(config)

crewai_adapter = CrewAIAdapter(config)
langgraph_adapter = LangGraphAdapter(config)

mlflow.set_tracking_uri(config.mlflow.tracking_uri)
mlflow.set_experiment("agent-evaluation")

print("Adapters initialized for evaluation")


In [ ]:
# Define evaluation tasks
EVALUATION_TASKS = [
    {
        "name": "simple-qa",
        "prompt": "What is the capital of France?",
        "expected_keywords": ["Paris"],
    },
    {
        "name": "math-problem",
        "prompt": "What is 25 * 17?",
        "expected_keywords": ["425"],
    },
    {
        "name": "reasoning",
        "prompt": "Explain why the sky is blue in one sentence.",
        "expected_keywords": ["scatter", "light", "atmosphere"],
    },
]

print(f"Defined {len(EVALUATION_TASKS)} evaluation tasks")


In [ ]:
# Evaluation helper function
import time

def evaluate_response(response: str, expected_keywords: list[str]) -> dict:
    """Score a response based on keyword matching."""
    response_lower = response.lower()
    matches = sum(1 for kw in expected_keywords if kw.lower() in response_lower)
    score = matches / len(expected_keywords) if expected_keywords else 0
    return {
        "score": score,
        "matches": matches,
        "total_keywords": len(expected_keywords),
    }

def run_evaluation(adapter_name: str, run_func, tasks: list) -> list[dict]:
    """Run evaluation tasks and log results."""
    results = []
    
    for task in tasks:
        with mlflow.start_run(run_name=f"{adapter_name}-{task['name']}"):
            mlflow.log_params({
                "adapter": adapter_name,
                "task": task["name"],
                "prompt": task["prompt"][:100],
            })
            
            start = time.time()
            try:
                response = run_func(task["prompt"])
                duration = time.time() - start
                
                eval_result = evaluate_response(str(response), task["expected_keywords"])
                
                mlflow.log_metrics({
                    "duration_seconds": duration,
                    "score": eval_result["score"],
                    "keyword_matches": eval_result["matches"],
                    "success": 1,
                })
                
                results.append({
                    "task": task["name"],
                    "score": eval_result["score"],
                    "duration": duration,
                })
                
            except Exception as e:
                mlflow.log_metric("success", 0)
                mlflow.set_tag("error", str(e)[:200])
                results.append({"task": task["name"], "score": 0, "error": str(e)})
    
    return results

print("Evaluation helpers ready")


In [ ]:
# Compare results across runs
runs = mlflow.search_runs(
    experiment_names=["agent-evaluation"],
    max_results=50,
)

if not runs.empty:
    print("Evaluation Results Summary:")
    print("-" * 50)
    
    summary = runs.groupby("params.adapter").agg({
        "metrics.score": ["mean", "std"],
        "metrics.duration_seconds": "mean",
    }).round(3)
    
    display(summary)
else:
    print("No evaluation runs found. Run the evaluation cells above first.")
